In [53]:
import torch

In [20]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
from sequential_models import Linear
dataset_path = "fra.txt"

Remove excess text (attributions) and non-break space chars

In [135]:
def preprocess_text(text):
  text = text.split("CC-BY")[0].strip()
  text = text.replace("\u202f", " ").replace("\xa0", " ")
  no_space = lambda char, prev_char: char in ".,?!" and prev_char != " "
  out = [" " + char if i > 0 and no_space(char, text[i-1]) else char
         for i, char in enumerate(text.lower())]
  result = "".join(out)
  return(result)

Word-level tokenization

In [136]:
def tokenize_data(text, src, tgt):
  parts = text.split("\t")
  if len(parts) == 2:
    src.append([t for t in f"{parts[0]} <eos>".split(" ") if t])
    tgt.append([t for t in f"{parts[1]} <eos>".split(" ") if t])



In [57]:
x = torch.tensor([True, False, False, True]).type(torch.int32).sum(0)
x

tensor(2)

In [143]:
src = []
tgt = []

Preprocess and tokenize the data

In [144]:
with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

Pad or truncate the text to obtain a regular shape for computationally efficient operations

In [145]:
def pad_or_truncate(text, num_steps=10):
  if len(text) >= num_steps:
    text = text[:num_steps]
  else:
    text += ["<pad>"] * (num_steps - len(text))
  return text


Lookup table

In [149]:
from collections import Counter

def Vocab(src=src, min_freq=2):

  def flatten(src=src):
    return [item for text in src for item in text]

  flattened_seq = flatten()
  counter = Counter(flattened_seq)
  vocab = {"<unk>":0}

  for token, freq in counter.items():
    if freq > min_freq:
      vocab[token] = freq
  return vocab

def encode(vocab, word):
  return vocab.get(word, vocab["<unk>"])



In [150]:
src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

In [152]:
print(src[0])
eng_vocab

['go', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


{'<unk>': 0,
 'go': 5322,
 '.': 181964,
 '<eos>': 205541,
 '<pad>': 540862,
 'hi': 29,
 'run': 409,
 '!': 1757,
 'who': 2772,
 '?': 36517,
 'wow': 16,
 'duck': 11,
 'fire': 361,
 'help': 2795,
 'hide': 115,
 'jump': 81,
 'stop': 1125,
 'wait': 802,
 'begin': 136,
 'on': 7358,
 'hello': 47,
 'i': 64757,
 'see': 2779,
 'try': 1015,
 'won': 174,
 'oh': 34,
 'no': 3738,
 'relax': 75,
 'shoot': 64,
 'smile': 98,
 'sorry': 1035,
 'attack': 65,
 'buy': 1046,
 'it': 14866,
 'cheers': 5,
 'eat': 1424,
 'exhale': 6,
 'get': 4323,
 'up': 3670,
 'now': 2361,
 'got': 2599,
 'hop': 4,
 'in': 13807,
 'hug': 55,
 'me': 13608,
 'fell': 365,
 'fled': 8,
 'hunt': 7,
 'knit': 12,
 'know': 8497,
 'left': 1187,
 'lied': 136,
 'lost': 793,
 'paid': 264,
 'pass': 166,
 'quit': 291,
 'swim': 385,
 "i'm": 11401,
 'ok': 321,
 'inhale': 5,
 'listen': 319,
 'way': 1501,
 'really': 2766,
 'thanks': 286,
 'we': 10195,
 'ask': 1008,
 'tom': 24272,
 'him': 4061,
 'awesome': 67,
 'be': 9550,
 'calm': 124,
 'cool': 111,